# NFL Player Stats Explorer — 2023 & 2024 Seasons

Exploratory analysis of NFL player stats using `nfl-data-py` and `pandas`. Data sourced from [nflverse](https://github.com/nflverse/nflverse-data).

**Sections:**
1. Setup & Data Load
2. Top QBs by Passing Yards (2024)
3. QB Efficiency - TD:INT Ratio
4. Top RBs by Rushing Yards (2024)
5. Year-over-Year RB Efficiency (2023 vs 2024)
6. Top WRs by Receiving Yards (2024)

## 1. Setup & Data Load

In [ ]:
import nfl_data_py as nfl
import pandas as pd

# Load seasonal stats for 2023 and 2024
df = nfl.import_seasonal_data([2023, 2024])

# Load player metadata for names, positions, teams
players_raw = nfl.import_players()
players = players_raw[['gsis_id', 'display_name', 'position', 'latest_team']]

print(f'Rows: {df.shape[0]} | Columns: {df.shape[1]}')
df.head()

## 2. Top QBs by Passing Yards (2024)

Filter to QBs with at least 100 pass attempts to exclude backups and spot starters.

In [ ]:
qbs = df[(df['season'] == 2024) & (df['attempts'] > 100)].copy()
qbs = qbs[['player_id', 'passing_yards', 'passing_tds', 'interceptions', 'attempts', 'passing_epa']]
qbs = qbs.sort_values('passing_yards', ascending=False)
qbs = qbs.merge(players, left_on='player_id', right_on='gsis_id', how='left')

qbs[['display_name', 'latest_team', 'passing_yards', 'passing_tds', 'interceptions', 'attempts']].head(10)

## 3. QB Efficiency - TD:INT Ratio & EPA

Passing yards alone don't tell the whole story. TD:INT ratio and Expected Points Added (EPA) per attempt are better efficiency metrics.

- **TD:INT ratio** - touchdowns per interception thrown
- **EPA per attempt** - how many points each pass attempt added on average vs. expected

In [ ]:
qbs['td_int_ratio'] = qbs['passing_tds'] / qbs['interceptions'].replace(0, 0.5)  # avoid divide by zero
qbs['epa_per_attempt'] = qbs['passing_epa'] / qbs['attempts']

efficiency = qbs[['display_name', 'latest_team', 'passing_tds', 'interceptions', 'td_int_ratio', 'epa_per_attempt']]
efficiency = efficiency.sort_values('epa_per_attempt', ascending=False)

efficiency.head(10)

## 4. Top RBs by Rushing Yards (2024)

Filter to RBs with at least 50 carries. Calculate yards per carry as a derived efficiency metric.

In [ ]:
rbs = df[(df['season'] == 2024) & (df['carries'] > 50)].copy()
rbs['yards_per_carry'] = rbs['rushing_yards'] / rbs['carries']
rbs = rbs[['player_id', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_epa', 'yards_per_carry']]
rbs = rbs.sort_values('rushing_yards', ascending=False)
rbs = rbs.merge(players, left_on='player_id', right_on='gsis_id', how='left')

rbs[['display_name', 'latest_team', 'rushing_yards', 'rushing_tds', 'carries', 'yards_per_carry']].head(10)

## 5. Year-over-Year RB Efficiency (2023 vs 2024)

Compare yards per carry for the top 10 rushers in 2024 against their 2023 numbers. Who improved? Who regressed?

In [ ]:
rbs23 = df[(df['season'] == 2023) & (df['carries'] > 50)].copy()
rbs23['yards_per_carry'] = rbs23['rushing_yards'] / rbs23['carries']

rbs24 = df[(df['season'] == 2024) & (df['carries'] > 50)].copy()
rbs24['yards_per_carry'] = rbs24['rushing_yards'] / rbs24['carries']

# Get top 10 rushers in 2024 by yards
top_ids = rbs24.sort_values('rushing_yards', ascending=False).head(10)['player_id'].tolist()

# Merge 2024 and 2023 side by side
comparison = rbs24[rbs24['player_id'].isin(top_ids)].merge(
    rbs23, on='player_id', suffixes=('_2024', '_2023'), how='left'
)

comparison['ypc_improvement'] = comparison['yards_per_carry_2024'] - comparison['yards_per_carry_2023']
comparison = comparison.merge(players, left_on='player_id', right_on='gsis_id', how='left')

comparison[[
    'display_name', 'latest_team',
    'rushing_yards_2024', 'rushing_yards_2023',
    'yards_per_carry_2024', 'yards_per_carry_2023',
    'ypc_improvement'
]].sort_values('ypc_improvement', ascending=False).round(2)

## 6. Top WRs by Receiving Yards (2024)

Filter to receivers with at least 50 targets. Calculate yards per reception and yards per target as efficiency metrics.

In [ ]:
wrs = df[(df['season'] == 2024) & (df['targets'] >= 50)].copy()
wrs['yards_per_reception'] = wrs['receiving_yards'] / wrs['receptions']
wrs['yards_per_target'] = wrs['receiving_yards'] / wrs['targets']
wrs['catch_rate'] = wrs['receptions'] / wrs['targets']

wrs = wrs[['player_id', 'targets', 'receptions', 'receiving_yards', 'receiving_tds',
           'yards_per_reception', 'yards_per_target', 'catch_rate']]
wrs = wrs.sort_values('receiving_yards', ascending=False)
wrs = wrs.merge(players, left_on='player_id', right_on='gsis_id', how='left')

wrs[['display_name', 'latest_team', 'receiving_yards', 'receiving_tds',
     'targets', 'receptions', 'yards_per_target', 'catch_rate']].head(10).round(2)